# Microstructure Simulation in Pure Julia
## Phase Fields with CALPHAD Coupling

This notebook demonstrates two Julia packages for computational materials science:

- **[OpenCALPHAD.jl](https://github.com/hsugawa8651/OpenCALPHAD.jl)**: CALPHAD thermodynamic calculation
- **[PhaseFields.jl](https://github.com/hsugawa8651/PhaseFields.jl)**: Phase field simulation

### Contents
1. Ag-Cu binary phase diagram (CALPHAD)
2. CALPHAD-driven phase field: driving force from thermodynamic database
3. 2D spinodal decomposition (Cahn-Hilliard)
4. Allen-Cahn 2D curvature-driven shrinkage
5. Thermal solidification with latent heat

In [ ]:
using Pkg
Pkg.add("OpenCALPHAD")
Pkg.add("PhaseFields")
Pkg.add("OrdinaryDiffEq")
Pkg.add("Plots")

In [ ]:
using OpenCALPHAD, PhaseFields, OrdinaryDiffEq, Plots
using Base64

# Helper: save GIF and display inline in Colab
function show_gif(anim, filename; fps=10)
    gif(anim, filename, fps=fps)
    data = base64encode(read(filename))
    display("text/html", """<img src="data:image/gif;base64,$data">""")
end

## Part 1: Ag-Cu Binary Phase Diagram

Calculate the FCC miscibility gap in the Ag-Cu system using CALPHAD thermodynamics.
The TDB (Thermodynamic DataBase) file contains Gibbs energy parameters.

In [ ]:
# Download Ag-Cu TDB from repository
tdb_url = "https://raw.githubusercontent.com/hsugawa8651/OpenCALPHAD.jl/main/reftest/tdb/agcu.TDB"
download(tdb_url, "agcu.TDB")

# Load TDB and compute phase diagram
db = read_tdb("agcu.TDB")
fcc = get_phase(db, "FCC_A1")
result = map_phase_diagram(fcc, db, 800.0, 1300.0, 25.0)

converged = filter(p -> p.converged, result.points)
T_vals = [p.temperature for p in converged]
x1_vals = [p.compositions[1] for p in converged]
x2_vals = [p.compositions[2] for p in converged]

In [ ]:
p = plot(x1_vals, T_vals, label="Cu-rich boundary",
    xlabel="x(Ag)", ylabel="Temperature [K]",
    title="Ag-Cu Phase Diagram (FCC Miscibility Gap)",
    lw=2, legend=:top, size=(700, 500))
plot!(p, x2_vals, T_vals, label="Ag-rich boundary", lw=2)

x_fill = vcat(x1_vals, reverse(x2_vals))
T_fill = vcat(T_vals, reverse(T_vals))
plot!(p, x_fill, T_fill, fillrange=minimum(T_vals),
    fillalpha=0.2, label="Two-phase region", lw=0)
p

## Part 2: CALPHAD-Driven Phase Field

Using the Ag-Cu binary system from Part 1, we compute the thermodynamic driving force between FCC solid and liquid phases.
The key advantage of Julia: the CALPHAD Gibbs energy is a plain Julia function,
so we can use **automatic differentiation** (AD) to compute chemical potentials
and driving forces directly from the thermodynamic database.

We then run a 1D Allen-Cahn simulation where the driving force is taken directly from CALPHAD at a fixed composition (x_Cu = 0.3, T = 1000K).

In [ ]:
# Driving force from CALPHAD at various compositions
x_plot = 0.05:0.01:0.6
ΔG_x = [driving_force(db, 1000.0, x, "FCC_A1", "LIQUID") for x in x_plot]

p1 = plot(title="Driving Force vs Composition (T=1000K)",
    xlabel="x_Cu", ylabel="ΔG [J/mol]", legend=:topright, size=(700, 400),
    bottom_margin=5Plots.mm, left_margin=3Plots.mm)
plot!(p1, x_plot, ΔG_x, label="ΔG (FCC - Liquid)", lw=2, color=:blue)
hline!([0], color=:gray, ls=:dash, label="")
x_solid = x_plot[ΔG_x .< 0]
if length(x_solid) > 0
    vspan!([minimum(x_solid), maximum(x_solid)], alpha=0.15, color=:blue, label="FCC stable")
end

# Driving force vs temperature at fixed composition
T_plot = 850:5:1250
ΔG_T = [driving_force(db, T, 0.3, "FCC_A1", "LIQUID") for T in T_plot]

p2 = plot(title="Driving Force vs Temperature (x_Cu=0.3)",
    xlabel="Temperature [K]", ylabel="ΔG [J/mol]", legend=:topleft, size=(700, 400),
    bottom_margin=5Plots.mm, left_margin=3Plots.mm)
plot!(p2, T_plot, ΔG_T, label="ΔG (FCC - Liquid)", lw=2, color=:red)
hline!([0], color=:gray, ls=:dash, label="")

plot(p1, p2, layout=(1, 2), size=(1100, 450))

In [ ]:
# Allen-Cahn with CALPHAD driving force
T_sim = 1000.0
x_sim = 0.3
ΔG_calphad = driving_force(db, T_sim, x_sim, "FCC_A1", "LIQUID")
println("CALPHAD driving force at T=$(T_sim)K, x_Cu=$(x_sim): ΔG = $(round(ΔG_calphad, digits=1)) J/mol")

model_calphad = AllenCahnModel(τ=1.0, W=1.0, m=1e-4)  # m scales ΔG to dimensionless
Nx_c = 100; dx_c = 1.0; dt_c = 0.1; Nt_c = 300
φ_c = [0.5 * (1 + tanh((i - 30) / 3)) for i in 1:Nx_c]

∇²φ_c = similar(φ_c)
x_grid_c = range(0, Nx_c*dx_c, length=Nx_c)
snaps_c = [(t=0, φ=copy(φ_c))]

function lap!(out, φ, dx)
    N = length(φ)
    for i in 2:N-1; out[i] = (φ[i+1] - 2φ[i] + φ[i-1]) / dx^2; end
    out[1] = (φ[2] - φ[1]) / dx^2
    out[N] = (φ[N-1] - φ[N]) / dx^2
end

for step in 1:Nt_c
    lap!(∇²φ_c, φ_c, dx_c)
    for i in 1:Nx_c
        dφdt = allen_cahn_rhs(model_calphad, φ_c[i], ∇²φ_c[i], ΔG_calphad)
        φ_c[i] = clamp(φ_c[i] + dt_c * dφdt, 0.0, 1.0)
    end
    if step % 5 == 0
        push!(snaps_c, (t=step, φ=copy(φ_c)))
    end
end
println("CALPHAD-driven simulation: $(length(snaps_c)) frames")

In [ ]:
anim_c = @animate for snap in snaps_c
    plot(x_grid_c, snap.φ, color=:blue, lw=2,
        fill=(0, 0.3, :blue), label="",
        xlabel="Position x", ylabel="φ (order parameter)",
        ylims=(-0.1, 1.1),
        title="CALPHAD-Driven Allen-Cahn (ΔG=$(round(Int, ΔG_calphad)) J/mol, t=$(snap.t))",
        size=(700, 400))
    hline!([0, 1], color=:gray, ls=:dash, label="")
end
show_gif(anim_c, "calphad_allen_cahn.gif", fps=15)

The plot shows the order parameter `phi` along a 1D domain: `phi = 1` represents the solid (FCC) phase and `phi = 0` represents the liquid phase.
The interface migrates to the right, driven by the CALPHAD driving force (`dG = -213 J/mol` at T=1000K, x_Cu=0.3), meaning the solid phase is thermodynamically favorable and grows at the expense of the liquid.

## Part 3: 2D Spinodal Decomposition

Cahn-Hilliard equation on a 2D grid: phase separation of a model binary A-B alloy from an unstable homogeneous mixture.
Unlike Parts 1-2 (Ag-Cu with CALPHAD), this example uses a generic double-well free energy with equilibrium compositions `c_alpha = 0.3` and `c_beta = 0.7`.
The composition variable `c` represents the mole fraction of component B.
Uses `OrdinaryDiffEq.jl` integration via the unified `PhaseFields.solve` API.

In [ ]:
# Cahn-Hilliard model (PFHub Benchmark 1 inspired)
ch_model = CahnHilliardModel(M=5.0, κ=2.0)
f = DoubleWellFreeEnergy(ρs=5.0, cα=0.3, cβ=0.7)

# 2D grid
Nx2, Ny2 = 128, 128
Lx, Ly = 200.0, 200.0
grid = UniformGrid2D(Nx=Nx2, Ny=Ny2, Lx=Lx, Ly=Ly)

# Initial condition: larger fluctuations for visible initial pattern
c0 = [0.5 + 0.05 * sin(4π * x / Lx) * cos(4π * y / Ly) +
      0.05 * 0.5 * sin(6π * x / Lx) * sin(6π * y / Ly)
      for x in grid.x, y in grid.y]

# Solve — long enough for clear coarsening
problem = CahnHilliardProblem(ch_model, grid, c0, (0.0, 100.0), f, bc=PeriodicBC())
sol = PhaseFields.solve(problem, ROCK2(), saveat=1.0, maxiters=500000)
println("Solved: $(length(sol.t)) time points, t_end = $(sol.t[end])")

In [ ]:
anim2 = @animate for idx in 1:length(sol.t)
    c = reshape(sol.u[idx], Nx2, Ny2)
    heatmap(grid.x, grid.y, c',
        c=:RdBu, clim=(0.2, 0.8),
        xlabel="x", ylabel="y",
        title="Spinodal Decomposition (t=$(round(sol.t[idx], digits=1)))",
        aspect_ratio=:equal, size=(500, 500))
end
show_gif(anim2, "spinodal.gif", fps=10)

The color map represents the local composition `c` (mole fraction of component B in a binary A-B system).
Starting from a nearly uniform mixture (`c = 0.5`), the system spontaneously separates into two equilibrium phases: the A-rich alpha phase (`c = 0.3`, blue) and the B-rich beta phase (`c = 0.7`, red).
This phase separation is driven by the double-well free energy, which has two minima at `c_alpha = 0.3` and `c_beta = 0.7`, corresponding to the equilibrium compositions of the two coexisting phases.

## Part 4: Allen-Cahn 2D — Curvature-Driven Shrinkage

A single-component system where a circular solid phase shrinks due to curvature-driven motion (no external driving force).
The order parameter `phi` represents solid (`phi = 1`) vs liquid (`phi = 0`), unlike the composition variable `c` in Part 3.
Uses `OrdinaryDiffEq.jl` via the unified `PhaseFields.solve` API.

In [ ]:
# Allen-Cahn 2D: circular solid shrinks by curvature
model_2d = AllenCahnModel(τ=1.0, W=0.05, m=0.0)
grid_2d = UniformGrid2D(Nx=100, Ny=100, Lx=1.0, Ly=1.0)

# Initial condition: circular solid at center (R=0.3)
φ0_2d = [sqrt((x - 0.5)^2 + (y - 0.5)^2) < 0.3 ? 1.0 : 0.0
         for x in grid_2d.x, y in grid_2d.y]

problem_2d = PhaseFieldProblem(
    model=model_2d, domain=grid_2d, φ0=φ0_2d,
    tspan=(0.0, 0.5), bc=NeumannBC())
sol_2d = PhaseFields.solve(problem_2d, Tsit5(), saveat=0.05)
println("Solved: $(length(sol_2d.t)) time points")

In [ ]:
anim3 = @animate for idx in 1:length(sol_2d.t)
    φ2d = reshape(sol_2d.u[idx], 100, 100)
    heatmap(grid_2d.x, grid_2d.y, φ2d',
        c=:viridis, clim=(0, 1),
        xlabel="x", ylabel="y",
        title="Allen-Cahn 2D: Curvature Shrinkage (t=$(round(sol_2d.t[idx], digits=2)))",
        aspect_ratio=:equal, size=(500, 500))
end
show_gif(anim3, "allen_cahn_2d.gif", fps=5)

The color map represents the order parameter `phi`: solid phase (`phi = 1`, yellow) and liquid phase (`phi = 0`, purple).
A circular solid region is initialized at the center. With no external driving force (`m = 0`), the interface moves inward driven solely by curvature, causing the solid circle to shrink over time.

## Part 5: Thermal Solidification with Latent Heat

A single-component system (Ni-like parameters) with coupled phase field and temperature evolution.
Unlike Part 4 (curvature-driven, no driving force), solidification here is driven by undercooling below the melting temperature.
A solid seed at center grows, releasing latent heat that raises the local temperature and slows further solidification.

In [ ]:
# Thermal solidification parameters (Nickel-like)
Tm = 1728.0         # Melting temperature [K]
L_latent = 2.35e9   # Latent heat [J/m³]
Cp = 5.42e6         # Heat capacity [J/(m³·K)]
α = 1e-5            # Thermal diffusivity [m²/s]
L_domain = 1e-4     # Domain length [m]
ΔT_undercool = 20.0 # Undercooling [K]

W_th = L_domain / 100
τ_th = W_th^2 / α * 0.5

thermal_model = ThermalPhaseFieldModel(
    τ=τ_th, W=W_th, λ=2.0, α=α, L=L_latent, Cp=Cp, Tm=Tm)

# Grid
Nx_th = 200
dx_th = L_domain / (Nx_th - 1)
x_th = collect(range(-L_domain/2, L_domain/2, length=Nx_th))

# Initial conditions
u_init = dimensionless_temperature(Tm - ΔT_undercool, Tm, L_latent, Cp)
seed_radius = 5 * W_th
φ_th = [0.5 * (1 - tanh((abs(xi) - seed_radius) / (sqrt(2) * W_th))) for xi in x_th]
u_th = fill(u_init, Nx_th)

# Time stepping
dt_th = thermal_stability_dt(thermal_model, dx_th)
t_end = 2e-3
n_steps = ceil(Int, t_end / dt_th)
dt_th = t_end / n_steps

# Storage
frame_interval = max(1, n_steps ÷ 50)
th_frames = [(t=0.0, φ=copy(φ_th),
    T=[physical_temperature(ui, Tm, L_latent, Cp) for ui in u_th])]

# Time evolution
for step in 1:n_steps
    ∇²φ_th = zeros(Nx_th)
    ∇²u_th = zeros(Nx_th)
    for i in 2:Nx_th-1
        ∇²φ_th[i] = (φ_th[i+1] - 2φ_th[i] + φ_th[i-1]) / dx_th^2
        ∇²u_th[i] = (u_th[i+1] - 2u_th[i] + u_th[i-1]) / dx_th^2
    end
    ∇²φ_th[1] = (φ_th[2] - φ_th[1]) / dx_th^2
    ∇²φ_th[Nx_th] = (φ_th[Nx_th-1] - φ_th[Nx_th]) / dx_th^2
    ∇²u_th[1] = (u_th[2] - u_th[1]) / dx_th^2
    ∇²u_th[Nx_th] = (u_th[Nx_th-1] - u_th[Nx_th]) / dx_th^2

    dφdt_th = [thermal_phase_rhs(thermal_model, φ_th[i], ∇²φ_th[i], u_th[i]) for i in 1:Nx_th]
    φ_new = clamp.(φ_th + dt_th * dφdt_th, 0.0, 1.0)
    dudt_th = [thermal_heat_rhs(thermal_model, u_th[i], ∇²u_th[i], dφdt_th[i]) for i in 1:Nx_th]
    u_th .= u_th + dt_th * dudt_th
    φ_th .= φ_new

    if step % frame_interval == 0
        T_snap = [physical_temperature(ui, Tm, L_latent, Cp) for ui in u_th]
        push!(th_frames, (t=step*dt_th, φ=copy(φ_th), T=T_snap))
    end
end
println("Thermal solidification: $(length(th_frames)) frames")

In [ ]:
x_μm = x_th * 1e6

anim4 = @animate for frame in th_frames
    t_ms = frame.t * 1000
    p1 = plot(x_μm, frame.φ, color=:blue, lw=2, fill=(0, 0.3, :blue),
        ylabel="φ", ylims=(-0.1, 1.1), legend=false,
        title="Thermal Solidification (t=$(round(t_ms, digits=2)) ms)")
    p2 = plot(x_μm, frame.T, color=:red, lw=2,
        xlabel="Position [μm]", ylabel="Temperature [K]",
        ylims=(Tm - ΔT_undercool - 5, Tm + 5), legend=false)
    hline!(p2, [Tm], color=:gray, ls=:dash)
    plot(p1, p2, layout=(2, 1), size=(700, 500))
end
show_gif(anim4, "thermal_solidification.gif", fps=8)

The upper panel shows the order parameter `phi`: solid (`phi = 1`) grows outward from a seed at the center into undercooled liquid (`phi = 0`).
The lower panel shows the temperature field: the dashed line marks the melting temperature (Tm = 1728 K). As the solid grows, latent heat is released at the interface, raising the local temperature toward Tm and decelerating the solidification front.